# 7.5 不算 loss 的內容就看不見嗎？

# 第 7 章：從續寫文字到回答問題

前置：第4章的因果模型與第1章的資料對齊。先用一筆不截斷對話，再回讀padding。完整render在tiny_perceptron/data.py；下列程式每節都新建資料，不依賴上一Notebook變量。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：兩種遮罩：看得見，不一定直接考它**

X 的 assistant 格預測 A；PAD 不當答案。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/masks.svg")))

**先想一想：**user embedding會因爲ignored label而永遠無梯度嗎？

沒有直接loss的user位置仍是後續attention的輸入。答案loss可以經由後面位置傳回user embedding；忽略監督不是切斷內容。

**把直覺寫成數學：**直接對該位置logits的梯度可爲0，但對它的輸入feature梯度仍可能非零。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.data import render_chat
from tiny_perceptron.model import TinyLM, ModelConfig, masked_loss

x, y = render_chat([{"role": "user", "content": "Q"}, {"role": "assistant", "content": "A"}])
model = TinyLM(ModelConfig(width=8))
result = model(x[None])
logits = result["logits"]
logits.retain_grad()
masked_loss(logits, y[None]).backward()
print(
    "第一格logits梯度",
    logits.grad[0, 0].norm().item(),
    "Q embedding梯度",
    model.embedding.weight.grad[x[2]].norm().item(),
)

**如何讀結果：**兩個梯度回答不同問題；不要只檢查一整層grad是否None。

**只改一件事：**只把user的Q替換成Z，保持答案一樣，觀察輸出變化。
